# Bootstrap Confidence Intervals & Paired Model Comparison

Loads post-Platt calibrated probabilities from `.npy` files and computes:
1. Per-model 95% bootstrap CIs for AUC-ROC, Average Precision, Brier score
2. Paired bootstrap CIs for all model-vs-model differences
3. Thesis-ready summary table

**Runtime:** ~60–90 seconds for 1,000 iterations over 2.1M test observations.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from pathlib import Path

In [2]:
# ── CONFIGURATION — adjust DATA_DIR if your .npy files live elsewhere ────────
DATA_DIR = Path("../data/predictions")
N_BOOT   = 1_000
SEED     = 42
ALPHA    = 0.05   # 95% CIs

In [3]:
# ── LOAD ARRAYS ──────────────────────────────────────────────────────────────
y_test   = np.load(DATA_DIR / "y_test.npy").astype(int)
cal_lr1  = np.load(DATA_DIR / "cal_proba_lr1.npy")   # LR  M1: weather only
cal_lr2  = np.load(DATA_DIR / "cal_proba_lr2.npy")   # LR  M2: weather + history
cal_lgb1 = np.load(DATA_DIR / "cal_proba_lgb1.npy")  # LGB M1: weather only
cal_lgb2 = np.load(DATA_DIR / "cal_proba_lgb2.npy")  # LGB M2: weather + history

assert len(y_test) == len(cal_lr1) == len(cal_lr2) == len(cal_lgb1) == len(cal_lgb2), \
    "Array lengths don't match — check your .npy files"

print(f"Test set: {len(y_test):,} observations | {y_test.sum():,} onsets "
      f"({y_test.mean()*100:.4f}%)")

Test set: 2,109,815 observations | 1,767 onsets (0.0838%)


In [4]:
# ── METRIC FUNCTION & POINT ESTIMATES ────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    return np.array([
        roc_auc_score(y_true, y_pred),
        average_precision_score(y_true, y_pred),
        brier_score_loss(y_true, y_pred)
    ])

METRIC_NAMES = ["AUC-ROC", "Avg Precision", "Brier Score"]

models = {
    "LR  M1 (weather only)": cal_lr1,
    "LR  M2 (weather+hist)": cal_lr2,
    "LGB M1 (weather only)": cal_lgb1,
    "LGB M2 (weather+hist)": cal_lgb2,
}

point_estimates = {name: compute_metrics(y_test, p) for name, p in models.items()}

print("Point estimates:")
for name, pe in point_estimates.items():
    print(f"  {name:35s}  AUC={pe[0]:.4f}  AP={pe[1]:.5f}  Brier={pe[2]:.6f}")

Point estimates:
  LR  M1 (weather only)                AUC=0.6624  AP=0.00149  Brier=0.000837
  LR  M2 (weather+hist)                AUC=0.7603  AP=0.00272  Brier=0.000836
  LGB M1 (weather only)                AUC=0.6226  AP=0.00131  Brier=0.000837
  LGB M2 (weather+hist)                AUC=0.7572  AP=0.00244  Brier=0.000836


In [5]:
# ── BOOTSTRAP — shared resample indices (required for valid paired test) ──────
rng          = np.random.default_rng(SEED)
n            = len(y_test)
MNAMES       = list(models.keys())
probas_list  = [cal_lr1, cal_lr2, cal_lgb1, cal_lgb2]
boot_metrics = np.zeros((N_BOOT, len(MNAMES), len(METRIC_NAMES)))

print(f"Running {N_BOOT:,} bootstrap iterations...")
for b in range(N_BOOT):
    idx = rng.integers(0, n, size=n)   # same idx for all 4 models — this is the paired part
    y_b = y_test[idx]
    if y_b.sum() == 0:
        boot_metrics[b] = np.nan
        continue
    for m, proba in enumerate(probas_list):
        boot_metrics[b, m] = compute_metrics(y_b, proba[idx])
    if (b + 1) % 100 == 0:
        print(f"  {b+1}/{N_BOOT}")

valid        = ~np.isnan(boot_metrics).any(axis=(1, 2))
boot_metrics = boot_metrics[valid]
print(f"Done. Used {valid.sum()} valid iterations.")

Running 1,000 bootstrap iterations...


  100/1000


  200/1000


  300/1000


  400/1000


...

  600/1000


  700/1000


  800/1000


  900/1000


  1000/1000
Done. Used 1000 valid iterations.


In [6]:
# ── TABLE 1: Per-model CIs ────────────────────────────────────────────────────
lo, hi = ALPHA / 2, 1 - ALPHA / 2
rows = []
for m, name in enumerate(MNAMES):
    pe = point_estimates[name]
    for k, metric in enumerate(METRIC_NAMES):
        rows.append({
            "Model":          name,
            "Metric":         metric,
            "Point estimate": round(pe[k], 6),
            "CI lower":       round(np.quantile(boot_metrics[:, m, k], lo), 6),
            "CI upper":       round(np.quantile(boot_metrics[:, m, k], hi), 6),
        })

df_metrics = pd.DataFrame(rows)

print("TABLE 1 — Per-model metrics with 95% bootstrap CIs\n")
for metric in METRIC_NAMES:
    print(f"  {metric}")
    sub = df_metrics[df_metrics["Metric"] == metric]
    for _, row in sub.iterrows():
        print(f"    {row['Model']:35s}  {row['Point estimate']:.4f}  "
              f"[{row['CI lower']:.4f}, {row['CI upper']:.4f}]")
    print()

TABLE 1 — Per-model metrics with 95% bootstrap CIs

  AUC-ROC
    LR  M1 (weather only)                0.6624  [0.6506, 0.6739]
    LR  M2 (weather+hist)                0.7603  [0.7494, 0.7712]
    LGB M1 (weather only)                0.6226  [0.6092, 0.6367]
    LGB M2 (weather+hist)                0.7572  [0.7461, 0.7679]

  Avg Precision
    LR  M1 (weather only)                0.0015  [0.0014, 0.0016]
    LR  M2 (weather+hist)                0.0027  [0.0025, 0.0031]
    LGB M1 (weather only)                0.0013  [0.0012, 0.0014]
    LGB M2 (weather+hist)                0.0024  [0.0022, 0.0027]

  Brier Score
    LR  M1 (weather only)                0.0008  [0.0008, 0.0009]
    LR  M2 (weather+hist)                0.0008  [0.0008, 0.0009]
    LGB M1 (weather only)                0.0008  [0.0008, 0.0009]
    LGB M2 (weather+hist)                0.0008  [0.0008, 0.0009]



In [7]:
# ── TABLE 2: Paired bootstrap differences ────────────────────────────────────
PAIRS = [
    ("LR  M1 (weather only)", "LGB M1 (weather only)", "LR M1 − LGB M1  (weather only)"),
    ("LR  M2 (weather+hist)", "LGB M2 (weather+hist)", "LR M2 − LGB M2  (weather+hist)"),
    ("LR  M2 (weather+hist)", "LR  M1 (weather only)", "LR M2 − LR M1   (history gain, LR)"),
    ("LGB M2 (weather+hist)", "LGB M1 (weather only)", "LGB M2 − LGB M1 (history gain, LGB)"),
]

pair_rows = []
print("TABLE 2 — Paired bootstrap differences, 95% CIs (AUC-ROC)")
print("  Positive = first model has higher AUC.")
print("  '** excludes 0' = statistically distinguishable.\n")

for name_a, name_b, label in PAIRS:
    idx_a = MNAMES.index(name_a)
    idx_b = MNAMES.index(name_b)

    diffs_auc = boot_metrics[:, idx_a, 0] - boot_metrics[:, idx_b, 0]
    diffs_ap  = boot_metrics[:, idx_a, 1] - boot_metrics[:, idx_b, 1]

    pe_auc = point_estimates[name_a][0] - point_estimates[name_b][0]
    pe_ap  = point_estimates[name_a][1] - point_estimates[name_b][1]

    ci_auc = (np.quantile(diffs_auc, lo), np.quantile(diffs_auc, hi))
    ci_ap  = (np.quantile(diffs_ap,  lo), np.quantile(diffs_ap,  hi))

    sig_auc = ci_auc[0] > 0 or ci_auc[1] < 0
    sig_ap  = ci_ap[0]  > 0 or ci_ap[1]  < 0

    print(f"  {label}")
    print(f"    AUC diff:  {pe_auc:+.4f}  [{ci_auc[0]:+.4f}, {ci_auc[1]:+.4f}]  "
          f"{'** excludes 0' if sig_auc else '(overlaps 0)'}")
    print(f"    AP  diff:  {pe_ap:+.4f}  [{ci_ap[0]:+.4f}, {ci_ap[1]:+.4f}]  "
          f"{'** excludes 0' if sig_ap else '(overlaps 0)'}")
    print()

    pair_rows.append({
        "Comparison": label,
        "AUC diff": pe_auc, "AUC CI lower": ci_auc[0], "AUC CI upper": ci_auc[1], "AUC sig.": sig_auc,
        "AP diff":  pe_ap,  "AP CI lower":  ci_ap[0],  "AP CI upper":  ci_ap[1],  "AP sig.":  sig_ap,
    })

df_pairs = pd.DataFrame(pair_rows)

TABLE 2 — Paired bootstrap differences, 95% CIs (AUC-ROC)
  Positive = first model has higher AUC.
  '** excludes 0' = statistically distinguishable.

  LR M1 − LGB M1  (weather only)
    AUC diff:  +0.0398  [+0.0283, +0.0499]  ** excludes 0
    AP  diff:  +0.0002  [+0.0001, +0.0003]  ** excludes 0

  LR M2 − LGB M2  (weather+hist)
    AUC diff:  +0.0030  [-0.0037, +0.0093]  (overlaps 0)
    AP  diff:  +0.0003  [+0.0001, +0.0005]  ** excludes 0

  LR M2 − LR M1   (history gain, LR)
    AUC diff:  +0.0979  [+0.0881, +0.1074]  ** excludes 0
    AP  diff:  +0.0012  [+0.0010, +0.0015]  ** excludes 0

  LGB M2 − LGB M1 (history gain, LGB)
    AUC diff:  +0.1346  [+0.1209, +0.1491]  ** excludes 0
    AP  diff:  +0.0011  [+0.0010, +0.0014]  ** excludes 0



In [8]:
# ── THESIS-READY SUMMARY TABLE ────────────────────────────────────────────────
print("THESIS TABLE — copy into Section 5\n")
header = f"{'Model':<36} {'AUC-ROC':>8}  {'95% CI':>17}  {'Avg Prec':>9}  {'95% CI':>17}  {'Brier':>9}"
print(header)
print("-" * len(header))

for m, name in enumerate(MNAMES):
    pe     = point_estimates[name]
    auc_lo = np.quantile(boot_metrics[:, m, 0], lo)
    auc_hi = np.quantile(boot_metrics[:, m, 0], hi)
    ap_lo  = np.quantile(boot_metrics[:, m, 1], lo)
    ap_hi  = np.quantile(boot_metrics[:, m, 1], hi)
    bs_lo  = np.quantile(boot_metrics[:, m, 2], lo)
    bs_hi  = np.quantile(boot_metrics[:, m, 2], hi)
    print(f"{name:<36} {pe[0]:>8.4f}  [{auc_lo:.4f},{auc_hi:.4f}]  "
          f"{pe[1]:>9.5f}  [{ap_lo:.5f},{ap_hi:.5f}]  {pe[2]:>9.6f}")

print("\nNotes: All probabilities post-Platt calibration. "
      "Bootstrap uses shared resample indices (paired test). "
      f"N={valid.sum()} valid iterations.")

THESIS TABLE — copy into Section 5

Model                                 AUC-ROC             95% CI   Avg Prec             95% CI      Brier
---------------------------------------------------------------------------------------------------------
LR  M1 (weather only)                  0.6624  [0.6506,0.6739]    0.00149  [0.00137,0.00164]   0.000837
LR  M2 (weather+hist)                  0.7603  [0.7494,0.7712]    0.00272  [0.00248,0.00308]   0.000836
LGB M1 (weather only)                  0.6226  [0.6092,0.6367]    0.00131  [0.00123,0.00142]   0.000837
LGB M2 (weather+hist)                  0.7572  [0.7461,0.7679]    0.00244  [0.00225,0.00272]   0.000836

Notes: All probabilities post-Platt calibration. Bootstrap uses shared resample indices (paired test). N=1000 valid iterations.


In [9]:
# ── SAVE ─────────────────────────────────────────────────────────────────────
df_metrics.to_csv(DATA_DIR / "bootstrap_metrics_ci.csv", index=False)
df_pairs.to_csv(DATA_DIR / "bootstrap_paired_diffs.csv", index=False)
np.save(DATA_DIR / "bootstrap_raw_metrics.npy", boot_metrics)
print(f"Saved to {DATA_DIR}:")
print("  bootstrap_metrics_ci.csv")
print("  bootstrap_paired_diffs.csv")
print("  bootstrap_raw_metrics.npy  (raw draws, for plotting)")

Saved to ../data/predictions:
  bootstrap_metrics_ci.csv
  bootstrap_paired_diffs.csv
  bootstrap_raw_metrics.npy  (raw draws, for plotting)
